<h3>3가지 청킹 × 2가지 임베딩 실험</h3>
<p>같은 교육용 원문과 질문을 유지하고 재귀·토큰·구조 기반 청킹과 두 임베딩 모델의 여섯 조합을 실제로 실행합니다. 이 작은 실험은 전체 검색 품질 순위를 정하지 않습니다.</p>

In [1]:
# 청킹·임베딩·검색 실험에 필요한 패키지를 설치합니다.
%pip install -q "python-dotenv==1.2.3" "langchain-core==1.6.2" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "tiktoken==0.14.0" "pypdf==6.18.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

from dotenv import find_dotenv, load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

env_file = find_dotenv(usecwd=True)
assert env_file, "실습 최상위 폴더에 .env를 준비하세요."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY를 확인하세요."
print("API 키 준비 완료")

API 키 준비 완료


<h3>1. 원문과 질문 고정하기</h3><p>제목 구조가 있는 배송·환불·회원 규정과 질문 하나를 모든 조합에서 그대로 사용합니다.</p>

In [3]:
text = """# 쇼핑몰 운영 규정

## 배송
배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.

## 환불
환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.

## 회원
회원 등급은 매월 갱신합니다. 포인트는 주문 금액의 1퍼센트가 적립됩니다.
"""
question = "제주 배송은 얼마나 더 걸리나요?"
print(text)
print("질문:", question)

# 쇼핑몰 운영 규정

## 배송
배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.

## 환불
환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.

## 회원
회원 등급은 매월 갱신합니다. 포인트는 주문 금액의 1퍼센트가 적립됩니다.

질문: 제주 배송은 얼마나 더 걸리나요?


<h3>2. 세 가지 청킹 실행하기</h3><p>재귀 청킹은 문자 경계를, 토큰 청킹은 토큰 수를, 구조 기반 청킹은 마크다운 제목을 우선 사용합니다.</p>

In [4]:
recursive = RecursiveCharacterTextSplitter(
    chunk_size=45, chunk_overlap=8
).split_text(text)
token = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=25, chunk_overlap=4
).split_text(text)
header_splitter = MarkdownHeaderTextSplitter(
    [("#", "제목"), ("##", "절")], strip_headers=False
)
structural_docs = header_splitter.split_text(text)
structural = [document.page_content for document in structural_docs]
chunk_sets = {"recursive": recursive, "token": token, "structural": structural}
for method, chunks in chunk_sets.items():
    print(method, "청크 수:", len(chunks))
    for index, chunk in enumerate(chunks):
        print(f"  {index:02d}", repr(chunk))

recursive 청크 수: 7
  00 '# 쇼핑몰 운영 규정'
  01 '## 배송'
  02 '배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.'
  03 '## 환불'
  04 '환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.'
  05 '## 회원'
  06 '회원 등급은 매월 갱신합니다. 포인트는 주문 금액의 1퍼센트가 적립됩니다.'
token 청크 수: 10
  00 '# 쇼핑몰 운영 규정'
  01 '## 배송'
  02 '배송은 결제 다음 날 시작합니다. 제주 배송은 기본'
  03 '기본 배송보다 이틀 더 걸립니다.'
  04 '## 환불'
  05 '환불은 수령 후 7일 이내 신청합니다. 사용한'
  06 '사용한 상품은 환불할 수 없습니다.'
  07 '## 회원'
  08 '회원 등급은 매월 갱신합니다. 포인트는 주문'
  09 '주문 금액의 1퍼센트가 적립됩니다.'
structural 청크 수: 3
  00 '# 쇼핑몰 운영 규정  \n## 배송\n배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.'
  01 '## 환불\n환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.'
  02 '## 회원\n회원 등급은 매월 갱신합니다. 포인트는 주문 금액의 1퍼센트가 적립됩니다.'


<h3>3. 여섯 조합에서 실제 임베딩 검색하기</h3><p>두 모델을 각각 세 개의 청크 집합에 적용합니다. 저장 공간 차이를 제외하고 검색 내용을 비교하기 위해 두 모델 모두 256차원으로 맞춥니다.</p>

In [5]:
models = ["text-embedding-3-small", "text-embedding-3-large"]
results = []
for method, chunks in chunk_sets.items():
    documents = [
        Document(page_content=chunk, metadata={"chunk_id": f"{method}:{i:02d}"})
        for i, chunk in enumerate(chunks)
    ]
    for model_name in models:
        embeddings = OpenAIEmbeddings(model=model_name, dimensions=256)
        store = InMemoryVectorStore.from_documents(documents, embeddings)
        hits = store.similarity_search_with_score(question, k=2)
        top_document, top_score = hits[0]
        results.append({
            "chunk_method": method,
            "embedding": model_name,
            "n_chunks": len(documents),
            "top_chunk_id": top_document.metadata["chunk_id"],
            "top_score": round(top_score, 4),
            "top_content": top_document.page_content,
        })
print("실행 수:", len(results))
assert len(results) == 6

실행 수: 6


<h3>4. 같은 형식으로 여섯 결과 읽기</h3><p>각 행에서 청킹 방법, 모델, 청크 수, 첫 검색 결과의 ID·점수·본문을 함께 확인합니다.</p>

In [6]:
for row in results:
    print(
        row["chunk_method"], "×", row["embedding"],
        "| chunks=", row["n_chunks"],
        "| top=", row["top_chunk_id"],
        "| score=", row["top_score"],
    )
    print(row["top_content"])

recursive × text-embedding-3-small | chunks= 7 | top= recursive:02 | score= 0.6487
배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.
recursive × text-embedding-3-large | chunks= 7 | top= recursive:02 | score= 0.6335
배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.
token × text-embedding-3-small | chunks= 10 | top= token:02 | score= 0.5625
배송은 결제 다음 날 시작합니다. 제주 배송은 기본
token × text-embedding-3-large | chunks= 10 | top= token:02 | score= 0.6148
배송은 결제 다음 날 시작합니다. 제주 배송은 기본
structural × text-embedding-3-small | chunks= 3 | top= structural:00 | score= 0.5769
# 쇼핑몰 운영 규정  
## 배송
배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.
structural × text-embedding-3-large | chunks= 3 | top= structural:00 | score= 0.5713
# 쇼핑몰 운영 규정  
## 배송
배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.


<h3>5. 첫 검색 결과의 근거 범위 확인하기</h3><p>첫 청크에 질문의 대상인 제주와 답인 이틀이 함께 있는지 확인합니다. 이 검사는 질문 하나의 관찰이며 검색 품질 지표가 아닙니다.</p>

In [7]:
for row in results:
    content = row["top_content"]
    row["contains_jeju_and_two_days"] = "제주" in content and "이틀" in content
    print(
        row["chunk_method"], "×", row["embedding"],
        "두 근거 포함:", row["contains_jeju_and_two_days"],
    )
print("질문 하나의 결과이므로 전체 검색 품질로 일반화하지 않습니다.")

recursive × text-embedding-3-small 두 근거 포함: True
recursive × text-embedding-3-large 두 근거 포함: True
token × text-embedding-3-small 두 근거 포함: False
token × text-embedding-3-large 두 근거 포함: False
structural × text-embedding-3-small 두 근거 포함: True
structural × text-embedding-3-large 두 근거 포함: True
질문 하나의 결과이므로 전체 검색 품질로 일반화하지 않습니다.


<h3>6. 제공 자료와 선택 실습 범위 확인하기</h3><p>배포 묶음의 <code>manifest.json</code>과 실제 PDF 파일을 대조합니다. 디지털 PDF 18건과 기존 문서의 스캔 변형 3건을 구분합니다. 집필용 전체 질의셋 149개는 정답 정보를 포함하므로 학생 배포에서 제외합니다. 이 자료를 수강생의 실제 업무 문서 100건이라고 세지 않습니다.</p>

In [8]:
import hashlib
import json
from pathlib import Path

root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "labs/data/rag-corpus").exists()
)
corpus = root / "labs/data/rag-corpus"
manifest = json.loads((corpus / "manifest.json").read_text(encoding="utf-8"))
corpus_sha = hashlib.sha256((corpus / "manifest.json").read_bytes()).hexdigest()
dataset_sha = hashlib.sha256((root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl").read_bytes()).hexdigest()
digital_rows = [row for row in manifest if row["tier"] != "scanned"]
scanned_rows = [row for row in manifest if row["tier"] == "scanned"]
missing = [row["file"] for row in manifest if not (corpus / row["file"]).exists()]
print("교육용 디지털 PDF:", len(digital_rows))
print("교육용 스캔 변형:", len(scanned_rows))
print("manifest와 실제 파일 불일치:", missing)
print("업무 문서 100건 준비 완료:", len(digital_rows) >= 100)
assert (len(digital_rows), len(scanned_rows), missing) == (18, 3, [])


교육용 디지털 PDF: 18
교육용 스캔 변형: 3
manifest와 실제 파일 불일치: []
업무 문서 100건 준비 완료: False


<h3>7. Ch 11과 같은 자료·질문으로 연결 검색하기</h3><p>앞의 짧은 원문 실험은 청킹 경계를 눈으로 비교하기 위한 학습용 범위입니다. 다음 셀은 Ch 11과 같은 18개 디지털 PDF와 Q001을 사용해 3가지 청킹×2가지 임베딩의 6개 설정을 실행합니다. 질문이 하나이므로 이 결과만으로 설정의 우열을 결정하지 않습니다.</p>

In [9]:
from pypdf import PdfReader
query_path = root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl"
query_rows = [json.loads(line) for line in query_path.read_text(encoding="utf-8").splitlines()]
query = next(row for row in query_rows if row["qid"] == "Q001")
pdf_paths = sorted(path for tier in ("easy", "medium", "hard") for path in (corpus / "pdf" / tier).glob("*.pdf"))
pages = []
for path in pdf_paths:
    for page, pdf_page in enumerate(PdfReader(path).pages, start=1):
        pages.append(Document(page_content=pdf_page.extract_text(), metadata={"doc_id": path.stem, "page": page}))
splitters = {"recursive": (RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100), {"chunk_size": 800, "chunk_overlap": 100}), "token": (RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name="cl100k_base", chunk_size=400, chunk_overlap=50), {"encoding_name": "cl100k_base", "chunk_size": 400, "chunk_overlap": 50}), "structural-page": (None, {"unit": "pdf_page"})}
continuity_runs = []
for method, (splitter, split_config) in splitters.items():
    chunks = pages if splitter is None else splitter.split_documents(pages)
    for number, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = f"ch10:{method}:{number:04d}"
    for model_name in models:
        embeddings = OpenAIEmbeddings(model=model_name, dimensions=256)
        store = InMemoryVectorStore.from_documents(chunks, embeddings)
        hits = store.similarity_search_with_score(query["question"], k=5)
        config = {"chunk_method": method, "split_config": split_config, "embedding": model_name, "dimensions": 256, "k": 5, "inputs": {"corpus_scope": "18 digital PDFs", "corpus_manifest_sha256": corpus_sha, "evaluation_dataset_sha256": dataset_sha}}
        continuity_runs.append({"config": config, "chunks": len(chunks), "hits": hits})
print("디지털 PDF:", len(pdf_paths), "쪽:", len(pages), "연결 실행:", len(continuity_runs))
print("Q001 설정별 1위:", [run["hits"][0][0].metadata["doc_id"] for run in continuity_runs])
assert len(continuity_runs) == 6 and query["qid"] == "Q001"


디지털 PDF: 18 쪽: 93 연결 실행: 6
Q001 설정별 1위: ['E01', 'E01', 'E01', 'E01', 'E01', 'E01']


<h3>8. 실행별 로그와 Ch 11 입력 포인터 저장하기</h3><p>실행 ID별 JSONL은 새 파일로 보존합니다. <code>continuity-ch10-indexing.jsonl</code>은 Ch 11이 읽는 최신 포인터이며 기존 Ch 10 로그를 덮어쓰지 않습니다.</p>

In [10]:
from datetime import UTC, datetime

base_run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%f-ch10")
runs_dir = root / "labs/part3/ch10-document-processing/runs"
archive_dir = runs_dir / "continuity"
archive_dir.mkdir(parents=True, exist_ok=True)
records = []
for number, run in enumerate(continuity_runs, start=1):
    source_run_id = f"{base_run_id}-{number:02d}"
    retrieved = [{"doc_id": doc.metadata["doc_id"], "chunk_id": doc.metadata["chunk_id"], "page": doc.metadata["page"], "rank": rank, "score": round(score, 6)} for rank, (doc, score) in enumerate(run["hits"], start=1)]
    record = {"qid": query["qid"], "question": query["question"], "gold_doc_ids": query["gold_doc_ids"], "source_chapter": "ch10", "source_run_id": source_run_id, "config": run["config"], "retrieved": retrieved}
    records.append(record)
    (archive_dir / f"{source_run_id}.jsonl").write_text(json.dumps(record, ensure_ascii=False) + "\n", encoding="utf-8")
lines = "".join(json.dumps(record, ensure_ascii=False) + "\n" for record in records)
pointer = runs_dir / "continuity-ch10-indexing.jsonl"
pointer.with_suffix(".tmp").write_text(lines, encoding="utf-8")
pointer.with_suffix(".tmp").replace(pointer)
saved = [json.loads(line) for line in pointer.read_text(encoding="utf-8").splitlines()]
print("실행별 보존 로그:", archive_dir.relative_to(root), "다음 챕터 입력:", pointer.relative_to(root))
print("저장한 설정:", len(saved), "Q001 1위 ID:", [row["retrieved"][0]["doc_id"] for row in saved])
assert saved == records


실행별 보존 로그: labs/part3/ch10-document-processing/runs/continuity 다음 챕터 입력: labs/part3/ch10-document-processing/runs/continuity-ch10-indexing.jsonl
저장한 설정: 6 Q001 1위 ID: ['E01', 'E01', 'E01', 'E01', 'E01', 'E01']


<h3>선택 실습</h3>
<p>수강생이 외부 전송이 허용된 실제 업무 문서 100건과 정답 질문을 준비한 뒤 같은 6개 조합으로 확장합니다. 파일 내용을 해시해 중복을 제외하고 100건인지 먼저 확인합니다. Upstage 또는 LlamaParse 비교는 각 서비스의 자격증명, 비용 확인, 문서 반출 승인이 모두 갖춰졌을 때 별도로 실행합니다. 이 기본 노트북은 상용 파서를 호출하지 않으며, 제공 PDF 21건이나 질문 149개를 업무 문서 100건으로 대체하지 않습니다.</p>